In [24]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import SGD
import matplotlib.pyplot as plt

In [25]:
NUM_CLIENTS=5
rounds=10
local_epochs=1
BATCH_SIZE=32
lr_rate=0.01

In [26]:
(X_train,Y_train),(X_test,Y_test)=mnist.load_data()

In [27]:
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0
X_train=X_train.reshape(-1,28*28)
X_test=X_test.reshape(-1,28*28)

In [28]:
def partition(x,y,n):
  idx=np.random.permutation(len(x))
  x=x[idx]
  y=y[idx]
  size=len(x)//n
  out=[]
  for i in range(n):
    s=i*size
    if(i<n-1):
      e=(i+1)*size
    else:
      e=len(x)
    out.append((x[s:e],y[s:e]))
  return out
clients_data=partition(X_train,Y_train,NUM_CLIENTS)

In [29]:
neighbours={}
for i in range(NUM_CLIENTS):
  neighbours[i]=[(i-1)%NUM_CLIENTS,(i+1)%NUM_CLIENTS]


In [30]:
def build_model():
    model=Sequential([
        Input(shape=(784,)),
        Dense(128,activation='relu'),
        Dense(64,activation='relu'),
        Dense(10,activation='softmax')
    ])
    model.compile(
        optimizer=SGD(learning_rate=lr_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [31]:
models=[build_model() for _ in range(NUM_CLIENTS)]

In [32]:
initial=models[0].get_weights()
for m in models:
  m.set_weights(initial)

In [33]:
def consensus(weights_list,ids):
  new=[]
  for layer in zip(*[weights_list[i] for i in ids]):
    new.append(np.mean(layer,axis=0))
  return new

In [34]:
for rnd in range(rounds):
  print(f"=======Round {rnd+1}=======")
  local_weights=[]
  for cid,(x,y) in enumerate(clients_data):
    models[cid].fit(x,y,epochs=local_epochs,batch_size=BATCH_SIZE,verbose=1)
    local_weights.append(models[cid].get_weights())
  updated=[]
  for cid in range(NUM_CLIENTS):
    ids=[cid]+neighbours[cid]
    updated.append(consensus(local_weights,ids))
  for cid in range(NUM_CLIENTS):
    models[cid].set_weights(updated[cid])
  acc=[]
  for m in models:
    _,a=m.evaluate(X_test,Y_test,verbose=1)
    acc.append(a)
  print("Average Test Accuracy:",np.mean(acc))
print("Training Complete.")

=======Round 1=======
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6451 - loss: 1.3819
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6464 - loss: 1.3967
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6467 - loss: 1.3920
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6428 - loss: 1.3981
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6472 - loss: 1.3906
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8448 - loss: 0.6829
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8469 - loss: 0.6820
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8467 - loss: 0.6840
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8457 - loss: 0.6851
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8464 - loss: 0.6832
Average Test Accuracy: 0.8461000084877014
=======Round 2=======
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8615 - loss: 0.5475
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8594 - loss: 0.5495
375/375 ━━━━━━